# Tooth Caries Classification (PyTorch Lightning + W&B)

This notebook performs binary classification on cropped tooth images using:

- Existing project data-preparation modules
- A custom PyTorch Lightning training loop
- Weights & Biases logging for loss, accuracy, and F1-metrics

In [17]:
from __future__ import annotations

import os
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision.models import resnet50, ResNet50_Weights

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
import torchmetrics

import wandb
from pathlib import Path
from dotenv import load_dotenv
from wandb.errors import AuthenticationError

# Ensure project imports work from notebook location.
PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
for p in (PROJECT_ROOT, SCRIPTS_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))


from classification_pipeline import (
    load_or_download_classification_dataset,
    ClassificationDownloadConfig,
    build_multiclass_classification_records_from_masks,
    split_grouped_records,
    ToothCropDataset,
    build_classification_image_pipeline,
    build_classification_resize_pipeline
)

In [32]:
@dataclass
class TrainConfig:
    image_size: int = 224
    batch_size: int = 1
    num_workers: int = 0
    max_epochs: int = 500
    lr: float = 1e-4
    weight_decay: float = 0.0
    wandb_project: str = 'tooth-caries-classification'
    wandb_run_name: str = 'resnet50 - last layer - overfit - fine tuning'
    force_download: bool = False

cfg = TrainConfig()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


In [33]:
class ToothClassificationDataModule(L.LightningDataModule):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.cfg = cfg
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def setup(self, stage: str | None = None):
        _, coco_data, image_dirs = load_or_download_classification_dataset(
            ClassificationDownloadConfig(api_key=os.getenv('ROBOFLOW_API_KEY')),
            force_download=self.cfg.force_download
        )
        
        all_records, self.label_map = build_multiclass_classification_records_from_masks(
            coco_data, image_dirs, crop_margin=0.08
        )
        
        train_rec, val_rec, test_rec = split_grouped_records(
            all_records, train_size=0.7, val_size=0.15, test_size=0.15
        )
        
        self.train_records_reduced = train_rec[:16] 
        self.val_records_reduced = val_rec[:16]
        self.test_records_reduced = test_rec[:16]
        
        aug_pipeline = build_classification_image_pipeline()
        resize_pipeline = build_classification_resize_pipeline(self.cfg.image_size)
        
        self.train_ds = ToothCropDataset(
            self.train_records_reduced,
            image_size=self.cfg.image_size,
            image_transform=aug_pipeline,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        self.val_ds = ToothCropDataset(
            self.val_records_reduced,
            image_size=self.cfg.image_size,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        self.test_ds = ToothCropDataset(
            self.test_records_reduced,
            image_size=self.cfg.image_size,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        
        print(f'Train samples: {len(self.train_ds)}\nVal samples: {len(self.val_ds)}\nTest samples: {len(self.test_ds)}')
        print(f'Train reduced samples: {len(self.train_records_reduced)}, Val reduced samples: {len(self.val_records_reduced)}, Test reduced samples: {len(self.test_records_reduced)}')
    
    def _loader_kwargs(self) -> dict[str, Any]:
        num_workers = int(self.cfg.num_workers)
        kwargs: dict[str, Any] = {
            'num_workers': num_workers,
            'pin_memory': torch.cuda.is_available(),
        }
        if num_workers > 0:
            kwargs['persistent_workers'] = True
            kwargs['prefetch_factor'] = 2
        return kwargs

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            **self._loader_kwargs()
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs()
        )
        
    def test_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs()
        )

In [34]:
class LitToothClassifier(L.LightningModule):
    def __init__(self, cfg, num_classes: int):
        super().__init__()
        
        self.cfg = cfg
        
        self.model = resnet50(weights=ResNet50_Weights.DEFAULT)
        
        #for param in self.model.parameters():
        #   param.requires_grad = False
            
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        
        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        
        self.log("train/loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        
        preds = torch.argmax(logits, dim=1)
        self.accuracy(preds, y)
        self.f1(preds, y)
        
        self.log("val/loss", loss, prog_bar=True, on_epoch=True)
        self.log("val/acc", self.accuracy, prog_bar=True, on_epoch=True)
        self.log("val/f1", self.f1, prog_bar=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.parameters()),
            lr=self.cfg.lr,
            weight_decay=self.cfg.weight_decay
        )
        return optimizer

In [35]:
env_path = Path('/work/.env')
if not env_path.exists():
    raise FileNotFoundError(f'.env file not found at {env_path}')

load_dotenv(env_path, override=True)

wandb_secret = (os.getenv('WANDB_API_KEY') or '').strip().strip('"').strip("'")
if not wandb_secret:
    raise RuntimeError(f'WANDB_API_KEY not found in {env_path}')

if len(wandb_secret) < 20:
    raise RuntimeError(
        f'Invalid WANDB_API_KEY length ({len(wandb_secret)}). '
        'Expected a classic API key or a wandb_v1 access token from https://wandb.ai/authorize.'
    )

# Support both classic API keys and wandb_v1 access tokens.
try:
    os.environ['WANDB_API_KEY'] = wandb_secret
    wandb.login(key=wandb_secret, relogin=True)
except AuthenticationError:
    if wandb_secret.startswith('wandb_v1_'):
        # Compatibility fallback for SDKs that validate only classic key shapes.
        compat_key = wandb_secret.replace('wandb_v1_', '', 1)
        os.environ['WANDB_API_KEY'] = compat_key
        wandb.login(key=compat_key, relogin=True)
    else:
        raise

print('W&B login successful using WANDB_API_KEY from .env')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B login successful using WANDB_API_KEY from .env


In [36]:
datamodule = ToothClassificationDataModule(cfg)
datamodule.setup()

model = LitToothClassifier(cfg=cfg, num_classes=len(datamodule.label_map))

wandb_logger = WandbLogger(
    project=cfg.wandb_project, 
    name=cfg.wandb_run_name,
    log_model=True
)

checkpoint_cb = ModelCheckpoint(
    dirpath=str(PROJECT_ROOT / 'output' / 'checkpoints' / 'classification'),
    filename='overfitting_1',
    monitor='val/f1',
    mode='max',
    save_top_k=2,
    save_last=True
)

trainer = L.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator='auto',
    devices=1,
    precision='16-mixed' if torch.cuda.is_available() else 32,
    logger=wandb_logger,
    callbacks=[
        checkpoint_cb, 
        LearningRateMonitor(logging_interval='epoch')
    ],
    log_every_n_steps=10,
    check_val_every_n_epoch=10
)

trainer.fit(model, datamodule=datamodule)

Train samples: 16
Val samples: 16
Test samples: 16
Train reduced samples: 16, Val reduced samples: 16, Test reduced samples: 16


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train samples: 16
Val samples: 16
Test samples: 16
Train reduced samples: 16, Val reduced samples: 16, Test reduced samples: 16


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model    │ ResNet             │ 23.7 M │ train │     0 │
│ 1 │ accuracy │ MulticlassAccuracy │      0 │ train │     0 │
│ 2 │ f1       │ MulticlassF1Score  │      0 │ train │     0 │
└───┴──────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 23.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.7 M                                                                                               
Total estimated model params size (MB): 94                                                                         
Modules in train mode: 153                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=500` reached.


In [8]:
datamodule.label_map

{'see': 0,
 '3M ESPE Implant': 1,
 '47': 2,
 '48': 3,
 '49': 4,
 '50': 5,
 '51': 6,
 '52': 7,
 '53': 8,
 '54': 9,
 '55': 10,
 '56': 11,
 '57': 12,
 '58': 13,
 '59': 14,
 '60': 15,
 '61': 16,
 '62': 17,
 '63': 18,
 '64': 19,
 '65': 20,
 '66': 21,
 '67': 22,
 '68': 23,
 '69': 24,
 '70': 25,
 '71': 26,
 '72': 27,
 '73': 28,
 '74': 29,
 '75': 30,
 '76': 31,
 '77': 32,
 '78': 33,
 'AGS Medikal Implant': 34,
 'AMerOss Implant': 35,
 'Amalgam filling': 36,
 'Anthogyr Implant': 37,
 'Bicon Implant': 38,
 'BioHorizons Implant': 39,
 'BioLife Implant': 40,
 'Biomet 3i Implant': 41,
 'Blue Sky Bio Implant': 42,
 'Camlog Implant': 43,
 'Caries': 44,
 'Composite filling': 45,
 'Cowellmedi Implant': 46,
 'Crown': 47,
 'DENTSPLY Implant': 48,
 'Dentatus Implant': 49,
 'Dentis Implant': 50,
 'Dentium Implant': 51,
 'Euroteknika Implant': 52,
 'Filling': 53,
 'Frontier Implant': 54,
 'Hiossen Implant': 55,
 'Implant': 56,
 'Implant Direct': 57,
 'Keystone Dental Implant': 58,
 'Leone Implant': 59,
 'MI